# Health Budget ML — Feature Engineering

This notebook creates separate feature-review datasets from the existing cleaned CSVs. It does not modify source files, invent values or dates, merge datasets, split data, or train models.

## Section 1 — Load Datasets

Locate the project root from the notebook/project structure and load only the three cleaned source files. For each dataset, report shape, columns, types, available years, and number of states. Source CSVs are read-only inputs.

In [9]:
from pathlib import Path

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "data" / "cleaned").is_dir()
        and (candidate / "notebooks").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        f"Could not locate project root from {working_directory}"
    )

notebook_directory = project_root / "notebooks"
cleaned_data_directory = (notebook_directory / ".." / "data" / "cleaned").resolve()
processed_data_directory = (notebook_directory / ".." / "data" / "processed").resolve()
dataset_paths = {
    "Budget": cleaned_data_directory / "nhm_budget_long_corrected.csv",
    "Infrastructure": cleaned_data_directory / "infrastructure_state_year.csv",
    "HMIS": cleaned_data_directory / "hmis_long_normalized.csv",
}

missing_paths = [path for path in dataset_paths.values() if not path.is_file()]
if missing_paths:
    missing_details = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(f"Required cleaned CSV file(s) are missing:\n{missing_details}")

raw_datasets = {
    dataset_name: pd.read_csv(path)
    for dataset_name, path in dataset_paths.items()
}
for dataset_name, dataset in raw_datasets.items():
    if dataset.empty or dataset.shape[1] == 0:
        raise ValueError(f"{dataset_name} did not load with rows and columns.")
    year_columns = [column for column in dataset.columns if column.casefold() == "year"]
    years = (
        sorted(dataset[year_columns[0]].dropna().astype(str).unique().tolist())
        if year_columns
        else []
    )
    state_columns = [column for column in dataset.columns if column.casefold() == "state"]
    number_of_states = (
        int(dataset[state_columns[0]].nunique(dropna=True)) if state_columns else None
    )
    print(
        f"{dataset_name}: shape={dataset.shape}; years={years or 'not present'}; "
        f"states={number_of_states if number_of_states is not None else 'not applicable'}"
    )
    print(f"Columns: {list(dataset.columns)}")
    print("Data types:")
    display(dataset.dtypes.rename("dtype").to_frame())

Budget: shape=(108, 5); years=['2021-22', '2022-23', '2023-24']; states=36
Columns: ['State', 'Year', 'Central_Release', 'Expenditure', 'Expenditure_to_Release_Pct']
Data types:


,dtype
State,str
Year,str
Central_Release,float64
Expenditure,float64
Expenditure_to_Release_Pct,float64


Infrastructure: shape=(74, 11); years=['2021-22', '2022-23']; states=38
Columns: ['State', 'Year', 'SHCs', 'PHCs', 'District_Hospitals', 'PHC_Beds', 'CHC_Beds', 'SDH_Beds', 'District_Hospital_Beds', 'Medical_College_Beds', 'Total_Beds']
Data types:


,dtype
State,str
Year,str
SHCs,float64
PHCs,float64
District_Hospitals,float64
PHC_Beds,float64
CHC_Beds,float64
SDH_Beds,float64
District_Hospital_Beds,float64
Medical_College_Beds,float64


HMIS: shape=(101840, 9); years=not present; states=37
Columns: ['Indicator', 'S.No.', 'Parameters', 'Type', 'State', 'Observation_Category', 'Value', 'Value_Numeric', 'Reporting_Period']
Data types:


,dtype
Indicator,str
S.No.,str
Parameters,str
Type,str
State,str
Observation_Category,str
Value,float64
Value_Numeric,float64
Reporting_Period,str


## Section 2 — Budget Feature Engineering

The budget table is the primary ML-ready candidate, with `Expenditure` retained as the target. Build numeric financial-year values and within-State lag features from real prior observations only. The lag columns are excluded from the target and, like every other feature, are built without future rows.

In [15]:
budget_source = raw_datasets["Budget"].copy()
budget_required_columns = {
    "State",
    "Year",
    "Central_Release",
    "Expenditure",
    "Expenditure_to_Release_Pct",
}
missing_budget_columns = sorted(budget_required_columns.difference(budget_source.columns))
if missing_budget_columns:
    raise ValueError(f"Budget dataset is missing required columns: {missing_budget_columns}")

budget_source["Year_Numeric"] = (
    budget_source["Year"]
    .astype("string")
    .str.extract(r"^(\d{4})-\d{2,4}$", expand=False)
    .astype("Int64")
)
invalid_budget_years = int(budget_source["Year_Numeric"].isna().sum())
print(f"Rows without a parseable YYYY-YY financial year: {invalid_budget_years}")
if invalid_budget_years:
    print("No year was inferred for these rows; their numeric year and lags remain missing.")

budget_state_year_duplicates = int(
    budget_source.duplicated(["State", "Year"], keep=False).sum()
)
print(f"Rows participating in duplicate State-Year keys: {budget_state_year_duplicates}")
if budget_state_year_duplicates:
    raise ValueError(
        "Duplicate State-Year rows prevent unambiguous within-State lags; "
        "no rows were removed."
    )

budget_ordered = budget_source.sort_values(
    ["State", "Year_Numeric"], kind="stable", na_position="last"
).reset_index(drop=True)
budget_groups = budget_ordered.groupby("State", sort=False, dropna=False)
budget_previous_year_numeric = budget_groups["Year_Numeric"].shift(1)
previous_expenditure = budget_groups["Expenditure"].shift(1)
previous_central_release = budget_groups["Central_Release"].shift(1)
consecutive_previous_year = (
    budget_ordered["Year_Numeric"] - budget_previous_year_numeric
).eq(1)

budget_features = budget_ordered[
    ["State", "Year", "Year_Numeric", "Central_Release", "Expenditure_to_Release_Pct"]
].copy()
budget_features["Previous_Year_Expenditure"] = previous_expenditure.where(
    consecutive_previous_year
)
budget_features["Previous_Year_Central_Release"] = previous_central_release.where(
    consecutive_previous_year
)
budget_features["Expenditure"] = budget_ordered["Expenditure"]

budget_predictor_columns = [
    "State",
    "Year",
    "Year_Numeric",
    "Central_Release",
    "Expenditure_to_Release_Pct",
    "Previous_Year_Expenditure",
    "Previous_Year_Central_Release",
]
budget_target_column = "Expenditure"
assert budget_target_column not in budget_predictor_columns
assert budget_features[budget_target_column].equals(budget_ordered[budget_target_column])

lag_rows = budget_features["Previous_Year_Expenditure"].notna()
assert (
    budget_ordered.loc[lag_rows, "Year_Numeric"].reset_index(drop=True)
    - budget_previous_year_numeric.loc[lag_rows].reset_index(drop=True)
).eq(1).all()
first_observation_rows = budget_previous_year_numeric.isna()
assert budget_features.loc[
    first_observation_rows,
    ["Previous_Year_Expenditure", "Previous_Year_Central_Release"],
].isna().all().all()

print(f"Budget feature dataframe shape: {budget_features.shape}")
print(f"Predictor columns: {budget_predictor_columns}")
print(f"Target column: {budget_target_column}")
print(f"Rows with actual consecutive prior-year lags: {int(lag_rows.sum())}")
print(f"Rows without a preceding observation: {int(first_observation_rows.sum())}")
print("Feature/target table preview:")
display(budget_features[budget_predictor_columns + [budget_target_column]].head(12))

processed_data_directory.mkdir(parents=True, exist_ok=True)
budget_output_path = processed_data_directory / "budget_features.csv"
budget_features.to_csv(budget_output_path, index=False)
print(f"Saved budget features to {budget_output_path}")

Rows without a parseable YYYY-YY financial year: 0
Rows participating in duplicate State-Year keys: 0
Budget feature dataframe shape: (108, 8)
Predictor columns: ['State', 'Year', 'Year_Numeric', 'Central_Release', 'Expenditure_to_Release_Pct', 'Previous_Year_Expenditure', 'Previous_Year_Central_Release']
Target column: Expenditure
Rows with actual consecutive prior-year lags: 72
Rows without a preceding observation: 36
Feature/target table preview:


,State,Year,Year_Numeric,Central_Release,Expenditure_to_Release_Pct,Previous_Year_Expenditure,Previous_Year_Central_Release,Expenditure
0,Andaman and Nicobar Islands,2021-22,2021,43.68,71.428571,NaN,NaN,31.20
1,Andaman and Nicobar Islands,2022-23,2022,45.26,93.415820,31.20,43.68,42.28
2,Andaman and Nicobar Islands,2023-24,2023,37.84,107.584567,42.28,45.26,40.71
3,Andhra Pradesh,2021-22,2021,1199.37,204.163019,NaN,NaN,2448.67
4,Andhra Pradesh,2022-23,2022,1489.45,128.526637,2448.67,1199.37,1914.34
5,Andhra Pradesh,2023-24,2023,1096.01,222.317315,1914.34,1489.45,2436.62
6,Arunachal Pradesh,2021-22,2021,188.53,131.814565,NaN,NaN,248.51
7,Arunachal Pradesh,2022-23,2022,233.82,181.669660,248.51,188.53,424.78
8,Arunachal Pradesh,2023-24,2023,404.55,98.736868,424.78,233.82,399.44
9,Assam,2021-22,2021,1955.93,112.190109,NaN,NaN,2194.36


Saved budget features to C:\Users\bagad\health-budget-ml\data\processed\budget_features.csv


## Section 3 — Infrastructure Feature Engineering

Create a separate feature table with State and Year preserved. Keep all original missing measurements as missing, add indicators for selected bed measures, and summarize per-variable year coverage. No imputation or forecasting claims are made.

In [11]:
infrastructure_source = raw_datasets["Infrastructure"].copy()
infrastructure_feature_columns = [
    "State",
    "Year",
    "SHCs",
    "PHCs",
    "District_Hospitals",
    "PHC_Beds",
    "CHC_Beds",
    "SDH_Beds",
    "District_Hospital_Beds",
    "Medical_College_Beds",
    "Total_Beds",
]
missing_infrastructure_columns = sorted(
    set(infrastructure_feature_columns).difference(infrastructure_source.columns)
)
if missing_infrastructure_columns:
    raise ValueError(
        "Infrastructure dataset is missing required columns: "
        f"{missing_infrastructure_columns}"
    )

infrastructure_features = infrastructure_source[infrastructure_feature_columns].copy()
infrastructure_features["Year_Numeric"] = (
    infrastructure_features["Year"]
    .astype("string")
    .str.extract(r"^(\d{4})-\d{2,4}$", expand=False)
    .astype("Int64")
)
for column in ["Total_Beds", "PHC_Beds", "CHC_Beds"]:
    infrastructure_features[f"{column}_Missing"] = (
        infrastructure_features[column].isna().astype("int8")
    )

infrastructure_duplicate_state_year_rows = int(
    infrastructure_features.duplicated(["State", "Year"], keep=False).sum()
)
print(f"Infrastructure feature dataframe shape: {infrastructure_features.shape}")
print(
    "Rows participating in duplicate State-Year keys: "
    f"{infrastructure_duplicate_state_year_rows}"
)
print("Original measurement missing values preserved; missingness indicators are added separately.")

temporal_coverage_rows = []
for column in infrastructure_feature_columns[2:]:
    observed = infrastructure_source.loc[
        infrastructure_source[column].notna(), ["State", "Year"]
    ]
    year_counts = observed.groupby("Year")["State"].nunique()
    temporal_coverage_rows.append(
        {
            "Variable": column,
            "Observed_Rows": len(observed),
            "Available_Years": ", ".join(sorted(observed["Year"].dropna().astype(str).unique())),
            "Number_of_Years": observed["Year"].nunique(),
            "Max_States_in_One_Year": int(year_counts.max()) if not year_counts.empty else 0,
            "Forecasting_Coverage": (
                "Insufficient: one observed year"
                if observed["Year"].nunique() < 2
                else "Limited: only two observed years"
            ),
        }
    )
infrastructure_temporal_coverage = pd.DataFrame(temporal_coverage_rows)
print("Temporal coverage by measurement (includes aggregate rows where present):")
display(infrastructure_temporal_coverage)
print("Infrastructure feature sample:")
display(infrastructure_features.head(8))

infrastructure_output_path = (
    processed_data_directory / "infrastructure_features.csv"
)
infrastructure_features.to_csv(infrastructure_output_path, index=False)
print(f"Saved infrastructure features to {infrastructure_output_path}")

Infrastructure feature dataframe shape: (74, 15)
Rows participating in duplicate State-Year keys: 0
Original measurement missing values preserved; missingness indicators are added separately.
Temporal coverage by measurement (includes aggregate rows where present):


,Variable,Observed_Rows,Available_Years,Number_of_Years,Max_States_in_One_Year,Forecasting_Coverage
0,SHCs,37,2021-22,1,37,Insufficient: one observed year
1,PHCs,37,2021-22,1,37,Insufficient: one observed year
2,District_Hospitals,37,2021-22,1,37,Insufficient: one observed year
3,PHC_Beds,37,2022-23,1,37,Insufficient: one observed year
4,CHC_Beds,36,2022-23,1,36,Insufficient: one observed year
5,SDH_Beds,29,2022-23,1,29,Insufficient: one observed year
6,District_Hospital_Beds,37,2022-23,1,37,Insufficient: one observed year
7,Medical_College_Beds,33,2022-23,1,33,Insufficient: one observed year
8,Total_Beds,37,2022-23,1,37,Insufficient: one observed year


Infrastructure feature sample:


,State,Year,SHCs,PHCs,District_Hospitals,PHC_Beds,CHC_Beds,SDH_Beds,District_Hospital_Beds,Medical_College_Beds,Total_Beds,Year_Numeric,Total_Beds_Missing,PHC_Beds_Missing,CHC_Beds_Missing
0,All India/Total,2022-23,NaN,NaN,NaN,148729.0,180384.0,111557.0,153815.0,224176.0,818661.0,2022,0,0,0
1,Andaman and Nicobar Islands,2021-22,124.0,27.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2021,1,1,1
2,Andaman and Nicobar Islands,2022-23,NaN,NaN,NaN,230.0,210.0,NaN,220.0,700.0,1360.0,2022,0,0,0
3,Andhra Pradesh,2021-22,11480.0,1689.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,2021,1,1,1
4,Andhra Pradesh,2022-23,NaN,NaN,NaN,8712.0,6400.0,5400.0,2600.0,12583.0,35695.0,2022,0,0,0
5,Arunachal Pradesh,2021-22,367.0,131.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,2021,1,1,1
6,Arunachal Pradesh,2022-23,NaN,NaN,NaN,397.0,594.0,NaN,1151.0,350.0,2492.0,2022,0,0,0
7,Assam,2021-22,4701.0,1010.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,2021,1,1,1


Saved infrastructure features to C:\Users\bagad\health-budget-ml\data\processed\infrastructure_features.csv


## Section 4 — HMIS Feature Review

Create a separate analytical subset from the existing HMIS columns. Since `Reporting_Period` is an undated snapshot label, do not create lags, rolling averages, future targets, or any other time-series features.

In [12]:
hmis_source = raw_datasets["HMIS"].copy()
hmis_feature_columns = [
    "Indicator",
    "S.No.",
    "Parameters",
    "Type",
    "State",
    "Observation_Category",
    "Value",
    "Value_Numeric",
    "Reporting_Period",
]
missing_hmis_columns = sorted(set(hmis_feature_columns).difference(hmis_source.columns))
if missing_hmis_columns:
    raise ValueError(f"HMIS dataset is missing required columns: {missing_hmis_columns}")
hmis_features_review = hmis_source[hmis_feature_columns].copy()

hmis_date_columns = [
    column
    for column in hmis_features_review.columns
    if column.casefold() in {"year", "month", "date"}
]
reporting_period_values = hmis_features_review["Reporting_Period"].dropna().unique().tolist()
parseable_reporting_date_rows = int(
    pd.to_datetime(
        hmis_features_review["Reporting_Period"],
        errors="coerce",
        format="mixed",
    )
    .notna()
    .sum()
)
print(f"HMIS review subset shape: {hmis_features_review.shape}")
print(f"HMIS Year/Month/Date columns: {hmis_date_columns or 'none'}")
print(f"Reporting_Period values: {reporting_period_values}")
print(f"Reporting_Period rows parseable as dates: {parseable_reporting_date_rows}")
print(
    "No time-series features or targets are created; historical dated HMIS "
    "data is required before disease forecasting features can be engineered."
)
print("Observation_Category counts:")
display(
    hmis_features_review["Observation_Category"]
    .value_counts(dropna=False)
    .rename("row_count")
    .to_frame()
)
print("Top 10 State labels by row count:")
display(
    hmis_features_review["State"]
    .value_counts(dropna=False)
    .head(10)
    .rename("row_count")
    .to_frame()
)

hmis_output_path = processed_data_directory / "hmis_features_review.csv"
hmis_features_review.to_csv(hmis_output_path, index=False)
print(f"Saved HMIS review subset to {hmis_output_path}")

HMIS review subset shape: (101840, 9)
HMIS Year/Month/Date columns: none
Reporting_Period values: ['Source snapshot (date not inferred)']
Reporting_Period rows parseable as dates: 0
No time-series features or targets are created; historical dated HMIS data is required before disease forecasting features can be engineered.
Observation_Category counts:


,row_count
Observation_Category,
Total [(A+B) or (C+D)],20368
Public [A],20368
Private [B],20368
Urban [C],20368
Rural [D],20368


Top 10 State labels by row count:


,row_count
State,
Dadra and Nagar Haveli and Daman and Diu,5360
Andaman and Nicobar Islands,2680
Andhra Pradesh,2680
Arunachal Pradesh,2680
Assam,2680
Bihar,2680
Chandigarh,2680
Chhattisgarh,2680
Delhi,2680


Saved HMIS review subset to C:\Users\bagad\health-budget-ml\data\processed\hmis_features_review.csv


## Section 5 — Feature Validation

Validate each processed table independently: dimensions, columns, missingness, duplicates, applicable State-Year keys, numeric types, and sample rows. For budget, verify that predictor columns exclude the current target, lags refer only to the immediately preceding observed financial year, and first observations keep missing lags.

In [17]:
processed_paths = {
    "Budget": budget_output_path,
    "Infrastructure": infrastructure_output_path,
    "HMIS": hmis_output_path,
}
processed_datasets = {
    dataset_name: pd.read_csv(path)
    for dataset_name, path in processed_paths.items()
}

for dataset_name, dataset in processed_datasets.items():
    print(f"## {dataset_name}: {dataset.shape}")
    print(f"Columns: {list(dataset.columns)}")
    missing_summary = dataset.isna().sum().rename("missing_count").to_frame()
    missing_summary["missing_percent"] = (
        dataset.isna().mean().mul(100).round(2) if len(dataset) else float("nan")
    )
    print("Missing-value percentages:")
    display(missing_summary)
    print(f"Exact duplicate rows: {int(dataset.duplicated().sum())}")
    if {"State", "Year"}.issubset(dataset.columns):
        duplicate_keys = int(
            dataset.duplicated(["State", "Year"], keep=False).sum()
        )
        print(f"Rows participating in duplicate State-Year keys: {duplicate_keys}")
    else:
        print("State-Year duplicate check: not applicable (no Year column).")
    print("Numeric data types:")
    display(dataset.select_dtypes(include="number").dtypes.rename("dtype").to_frame())
    print("First 5 rows:")
    display(dataset.head(5))

assert budget_target_column == "Expenditure"
assert budget_target_column not in budget_predictor_columns
saved_budget = processed_datasets["Budget"]
saved_budget_lag_rows = saved_budget["Previous_Year_Expenditure"].notna()
saved_previous_year = budget_previous_year_numeric.loc[saved_budget_lag_rows]
saved_current_year = saved_budget.loc[saved_budget_lag_rows, "Year_Numeric"]
assert (saved_previous_year < saved_current_year).all()
assert (saved_current_year - saved_previous_year).eq(1).all()
assert saved_budget.loc[
    first_observation_rows,
    ["Previous_Year_Expenditure", "Previous_Year_Central_Release"],
].isna().all().all()
pd.testing.assert_series_equal(
    saved_budget["Previous_Year_Expenditure"],
    budget_features["Previous_Year_Expenditure"],
    check_dtype=False,
    check_names=False,
)
print("Budget leakage check: passed; predictors exclude current Expenditure target.")
print("Budget lag check: passed; lag rows use only the actual immediately prior year.")
print("First-year lag check: passed; rows without a previous year retain NaN lags.")

## Budget: (108, 8)
Columns: ['State', 'Year', 'Year_Numeric', 'Central_Release', 'Expenditure_to_Release_Pct', 'Previous_Year_Expenditure', 'Previous_Year_Central_Release', 'Expenditure']
Missing-value percentages:


,missing_count,missing_percent
State,0,0.00
Year,0,0.00
Year_Numeric,0,0.00
Central_Release,0,0.00
Expenditure_to_Release_Pct,0,0.00
Previous_Year_Expenditure,36,33.33
Previous_Year_Central_Release,36,33.33
Expenditure,0,0.00


Exact duplicate rows: 0
Rows participating in duplicate State-Year keys: 0
Numeric data types:


,dtype
Year_Numeric,int64
Central_Release,float64
Expenditure_to_Release_Pct,float64
Previous_Year_Expenditure,float64
Previous_Year_Central_Release,float64
Expenditure,float64


First 5 rows:


...

Exact duplicate rows: 1539
State-Year duplicate check: not applicable (no Year column).
Numeric data types:


,dtype
Value,float64
Value_Numeric,float64


First 5 rows:


,Indicator,S.No.,Parameters,Type,State,Observation_Category,Value,Value_Numeric,Reporting_Period
0,M1 [Ante Natal Care (ANC)],'1.1',Total number of pregnant women registered for ANC,TOTAL,Andaman and Nicobar Islands,Total [(A+B) or (C+D)],414.0,414.0,Source snapshot (date not inferred)
1,M1 [Ante Natal Care (ANC)],'1.1.1',"Out of the total ANC registered, number regist...",TOTAL,Andaman and Nicobar Islands,Total [(A+B) or (C+D)],317.0,317.0,Source snapshot (date not inferred)
2,M1 [Ante Natal Care (ANC)],'1.2.1',Number of PW given TT1,TOTAL,Andaman and Nicobar Islands,Total [(A+B) or (C+D)],328.0,328.0,Source snapshot (date not inferred)
3,M1 [Ante Natal Care (ANC)],'1.2.2',Number of PW given TT2,TOTAL,Andaman and Nicobar Islands,Total [(A+B) or (C+D)],199.0,199.0,Source snapshot (date not inferred)
4,M1 [Ante Natal Care (ANC)],'1.2.3',Number of PW given TT Booster,TOTAL,Andaman and Nicobar Islands,Total [(A+B) or (C+D)],57.0,57.0,Source snapshot (date not inferred)


Budget leakage check: passed; predictors exclude current Expenditure target.
Budget lag check: passed; lag rows use only the actual immediately prior year.
First-year lag check: passed; rows without a previous year retain NaN lags.


## Section 6 — Feature Engineering and ML Readiness

### Budget
- **Target:** `Expenditure`.
- **Available features:** `State`, original financial `Year`, `Year_Numeric`, `Central_Release`, and `Expenditure_to_Release_Pct`.
- **Lag features:** `Previous_Year_Expenditure` and `Previous_Year_Central_Release`, sourced only from each State's actual immediately preceding financial year. The first observation per State retains missing lag values.
- **Limitation:** Only three annual periods are available. The feature table is prepared for later review; this history alone does not establish reliable forecasting performance.

### Infrastructure
- **Available features:** `State`, original `Year`, `Year_Numeric`, the nine facility/bed measures, and selected missingness indicators.
- **Missingness and coverage:** Original measurements remain missing where absent. Every facility/bed measure has data for only one year in the current file, so additional historical coverage is required before forecasting.

### HMIS
- **Time-series features:** None created. `Reporting_Period` contains only `Source snapshot (date not inferred)` and there are no usable year/month/date columns.
- **Additional dated data required:** Historical observations with documented reporting dates and stable indicator, location, and category identifiers. The current file does not support disease time-series forecasting.

All feature tables remain separate. No cleaned CSV was modified, no rows were removed, and no model or train/test split was created.

In [18]:
ml_readiness_summary = pd.DataFrame(
    [
        {
            "Dataset": "Budget",
            "Processed File": "budget_features.csv",
            "Target": "Expenditure",
            "ML Status": "Review required",
            "Main Limitation": "Only three annual periods; forecasting needs more history.",
        },
        {
            "Dataset": "Infrastructure",
            "Processed File": "infrastructure_features.csv",
            "Target": "No target selected",
            "ML Status": "Needs additional data",
            "Main Limitation": "Each facility/bed measure has only one observed year.",
        },
        {
            "Dataset": "HMIS",
            "Processed File": "hmis_features_review.csv",
            "Target": "No time-series target created",
            "ML Status": "Needs additional dated data",
            "Main Limitation": "No reliable year/month/date in Reporting_Period.",
        },
    ]
)
display(ml_readiness_summary)

,Dataset,Processed File,Target,ML Status,Main Limitation
0,Budget,budget_features.csv,Expenditure,Review required,Only three annual periods; forecasting needs m...
1,Infrastructure,infrastructure_features.csv,No target selected,Needs additional data,Each facility/bed measure has only one observe...
2,HMIS,hmis_features_review.csv,No time-series target created,Needs additional dated data,No reliable year/month/date in Reporting_Period.
